# Probe del LLM real — ¿evita las falsas alarmas?

**Fase 5 · Tesis PR-196.** En `covariate` y `mezcla` NO hay daño real (el AUC se mantiene), pero el PSI se dispara. Las reglas fijas (y el LLM simulado) sobre-actúan: **falsas alarmas**.

La pregunta: ¿el **LLM real** reconoce que es benigno (*PSI alto pero AUC sano*) y decide **esperar**? Ese es el aporte de IA en el caso difícil.

Probe **barato y dirigido**: no una simulación completa, sino el LLM decidiendo en los **meses clave**. Con **caché**: la 1ª vez cuesta ~centavos, las siguientes son gratis.

## Preparación y carga

In [ ]:
import os, sys
from pathlib import Path

raiz = Path.cwd()
while not (raiz / "src").exists():
    raiz = raiz.parent
os.chdir(raiz)
sys.path.insert(0, str(raiz / "src"))

import pandas as pd
from tesis.modelo_base import ModeloBase
from tesis.simulador.poblacion import cargar_poblacion, COHORTE_OOT
from tesis.simulador.simulador import ConfigSimulacion, Simulador
from tesis.senales.monitor import Referencia, MonitorSenales
from tesis.politicas import Contexto, PoliticaLLM, guardar_cache, cargar_cache
from tesis.experimentos.matriz import construir_plan

modelo = ModeloBase.cargar("models/modelo_base.pkl")
pool = cargar_poblacion(cohorte=COHORTE_OOT)
dev = cargar_poblacion(cohorte=None); dev = dev[dev["FECHA_CORTE"] < COHORTE_OOT]
ref = Referencia.construir(dev, modelo)

def reportes(esc, N=26):
    """Corre el monitor (gratis) y devuelve [(reporte, contexto)] mes a mes."""
    plan = construir_plan(esc, N)
    cfg = ConfigSimulacion(n_periodos=N, tam_lote=2000, semilla=13579)
    mon = MonitorSenales(ref); pares = []
    for lote in Simulador(pool, cfg, plan).stream():
        r = mon.observar(lote, modelo)
        pares.append((r, Contexto(historia=mon.reportes[:-1])))
    return pares
print("listo")

## El LLM real decide en los meses clave

Meses donde el PSI está alto **y** el AUC tardío ya confirma que el modelo está sano → la trampa de la falsa alarma. Lo correcto es **esperar**.

Con caché por escenario: si ya se corrió, re-lee sin llamar a la API (costo 0).

In [ ]:
pol = PoliticaLLM(modelo="claude-haiku-4-5")   # LLM real, el más barato

filas = []
for esc in ["covariate", "mezcla"]:
    ruta = f"experiments/probe_{esc}_haiku.json"
    pol.cache = cargar_cache(ruta)              # replay gratis si ya se corrió
    pares = reportes(esc)
    clave = [(r, c) for (r, c) in pares
             if r.psi_score >= 0.15 and r.auc_revelado is not None and r.auc_revelado >= 0.85][:3]
    for r, c in clave:
        dec = pol.decidir(r, c)
        filas.append({"escenario": esc, "mes": r.periodo, "psi_score": round(r.psi_score, 3),
                      "auc_tardio": round(r.auc_revelado, 3), "decision": dec.accion.value,
                      "razon": dec.razon[:160]})
    guardar_cache(pol.cache, ruta)              # graba el trace de esta corrida

print(f"costo de esta corrida: USD {pol.costo_estimado(1.0, 5.0)}  |  llamadas a la API: {pol.uso['llamadas']}")
print("(si es 0, todo salió de caché → gratis)")
pd.DataFrame(filas)

## Lectura

Si la columna `decision` es **esperar** en todas: el LLM real **reconoce que el drift es benigno** (PSI alto pero AUC sano) y evita la falsa alarma — justo donde el simulado (regla cruda) sobre-actuaba (13 acciones en mezcla). Es el **aporte de IA** demostrado en el caso difícil.

Lee la columna `razon` para ver **cómo lo justifica** (menciona el AUC sano y el drift focalizado).